In [1]:


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import warnings
warnings.filterwarnings('ignore')

from scipy.io import arff
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, f1_score,
    precision_recall_curve, average_precision_score
)
from sklearn.feature_selection import SelectFromModel
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier



ModuleNotFoundError: No module named 'matplotlib'

In [ ]:

data, meta = arff.loadarff('5year.arff')
df = pd.DataFrame(data)

# Convert target column from bytes to integer
df['class'] = df['class'].astype(int)

print(f'Dataset shape: {df.shape}')
print(f'\nFeatures: {df.shape[1] - 1}')
print(f'Samples : {df.shape[0]}')
print('\nClass distribution (0 = Healthy, 1 = Bankrupt):')
print(df['class'].value_counts())
print(f'\nClass imbalance ratio: {round(df["class"].value_counts()[0] / df["class"].value_counts()[1], 1)}:1')
df.head()

In [ ]:
# --- Check missing values ---
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

print('Columns with missing values:')
print(missing_df)
print(f'\nTotal missing values: {df.isnull().sum().sum()}')

# Plot missing values
plt.figure(figsize=(12, 4))
missing_df['Missing %'].plot(kind='bar', color='salmon')
plt.title('Missing Values Per Feature (%)')
plt.ylabel('Missing %')
plt.xlabel('Feature')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# --- Strategy: Drop columns with >40% missing, fill rest with median ---

# Drop high-missing columns
threshold = 40
cols_to_drop = missing_df[missing_df['Missing %'] > threshold].index.tolist()
print(f'Dropping columns with >{threshold}% missing: {cols_to_drop}')
df_clean = df.drop(columns=cols_to_drop)

# Fill remaining missing values with column median
# Median is more robust than mean for financial data (handles outliers)
df_clean = df_clean.fillna(df_clean.median(numeric_only=True))

print(f'\nShape after cleaning: {df_clean.shape}')
print(f'Remaining missing values: {df_clean.isnull().sum().sum()}')

In [ ]:
# --- Handle extreme outliers using IQR capping ---
# Cap outliers at 1st and 99th percentile (Winsorization)

X_raw = df_clean.drop('class', axis=1)
y = df_clean['class']

for col in X_raw.columns:
    p1  = X_raw[col].quantile(0.01)
    p99 = X_raw[col].quantile(0.99)
    X_raw[col] = X_raw[col].clip(lower=p1, upper=p99)

print('✅ Outlier capping complete (1st-99th percentile)')
print(f'Features available: {X_raw.shape[1]}')

In [ ]:
# Class distribution pie chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pie chart
y.value_counts().plot.pie(
    ax=axes[0],
    labels=['Healthy (0)', 'Bankrupt (1)'],
    autopct='%1.1f%%',
    colors=['#2ecc71', '#e74c3c'],
    startangle=90
)
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('')

# Feature distributions for first 6 features split by class
axes[1].axis('off')
fig.suptitle('Dataset Overview', fontsize=14)
plt.tight_layout()
plt.show()

# Distribution of key financial features by class
key_features = ['Attr1', 'Attr2', 'Attr3', 'Attr5', 'Attr6', 'Attr9']
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, feat in enumerate(key_features):
    if feat in X_raw.columns:
        X_raw[y==0][feat].hist(ax=axes[i], bins=40, alpha=0.6, color='#2ecc71', label='Healthy')
        X_raw[y==1][feat].hist(ax=axes[i], bins=40, alpha=0.6, color='#e74c3c', label='Bankrupt')
        axes[i].set_title(f'{feat} Distribution')
        axes[i].legend()

plt.suptitle('Feature Distributions by Class', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# --- Scale features ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)
X_scaled = pd.DataFrame(X_scaled, columns=X_raw.columns)

print('✅ Features scaled with StandardScaler')

# --- Feature Selection using Random Forest importance ---
rf_selector = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_selector.fit(X_scaled, y)

# Get feature importances
importances = pd.Series(rf_selector.feature_importances_, index=X_raw.columns)
importances = importances.sort_values(ascending=False)

# Select top 20 features
top_features = importances.head(20).index.tolist()
X_selected = X_scaled[top_features]

print(f'\nFeatures reduced: {X_scaled.shape[1]} → {X_selected.shape[1]}')
print('\nTop 20 selected features:')
print(top_features)

# Plot feature importances
plt.figure(figsize=(12, 6))
importances.head(20).plot(kind='bar', color='steelblue')
plt.title('Top 20 Feature Importances (Random Forest)')
plt.ylabel('Importance Score')
plt.xlabel('Feature')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Split data BEFORE applying SMOTE (important: only apply SMOTE to training data)
X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y, test_size=0.2, random_state=42, stratify=y
)

print('Before SMOTE:')
print(f'  Training set: {X_train.shape[0]} samples')
print(f'  Class distribution: {dict(y_train.value_counts())}')

# Apply SMOTE only to training data
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print('\nAfter SMOTE:')
print(f'  Training set: {X_train_res.shape[0]} samples')
print(f'  Class distribution: {dict(pd.Series(y_train_res).value_counts())}')
print(f'\n  Test set (untouched): {X_test.shape[0]} samples')

In [ ]:
# Define 3 models to compare
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'XGBoost':             XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss', verbosity=0)
}

results = {}

for name, model in models.items():
    model.fit(X_train_res, y_train_res)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    results[name] = {
        'model':  model,
        'y_pred': y_pred,
        'y_prob': y_prob,
        'f1':     f1_score(y_test, y_pred, average='weighted'),
        'roc_auc': roc_auc_score(y_test, y_prob)
    }
    print(f'\n📌 {name}')
    print(f'   F1 Score : {results[name]["f1"]:.4f}')
    print(f'   ROC-AUC  : {results[name]["roc_auc"]:.4f}')

In [ ]:
# --- Detailed report for best model (XGBoost typically best) ---
best_model_name = max(results, key=lambda x: results[x]['roc_auc'])
print(f'🏆 Best Model: {best_model_name}')
print('\nClassification Report:')
print(classification_report(y_test, results[best_model_name]['y_pred'],  target_names=['Healthy', 'Bankrupt']))

# Confusion matrix
cm = confusion_matrix(y_test, results[best_model_name]['y_pred'])
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Healthy', 'Bankrupt'],
            yticklabels=['Healthy', 'Bankrupt'])
plt.title(f'Confusion Matrix — {best_model_name}')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

In [ ]:
# --- ROC Curves for all models ---
plt.figure(figsize=(8, 6))

for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    plt.plot(fpr, tpr, label=f"{name} (AUC = {res['roc_auc']:.3f})")

plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — Model Comparison')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Use the best model for SHAP explanations
best_model = results[best_model_name]['model']

# Create SHAP explainer
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test)

# For binary classification, use values for class 1 (Bankrupt)
if isinstance(shap_values, list):
    sv = shap_values[1]  # Random Forest returns list
else:
    sv = shap_values     # XGBoost returns array

print(f'✅ SHAP values computed for {best_model_name}')
print(f'   SHAP array shape: {sv.shape}')

In [ ]:
# --- SHAP Summary Plot (Global Feature Importance) ---
# Shows which features matter most ACROSS ALL predictions
plt.figure(figsize=(10, 7))
shap.summary_plot(sv, X_test, plot_type='bar', show=False)
plt.title('SHAP Global Feature Importance — Which features drive bankruptcy prediction?')
plt.tight_layout()
plt.show()

In [ ]:
# --- SHAP Beeswarm Plot (Direction of Impact) ---
# Shows HOW each feature pushes predictions toward healthy or bankrupt
# Red = high feature value, Blue = low feature value
plt.figure(figsize=(10, 7))
shap.summary_plot(sv, X_test, show=False)
plt.title('SHAP Beeswarm — How feature values affect bankruptcy risk')
plt.tight_layout()
plt.show()

In [ ]:
# --- SHAP Waterfall Plot (Single Prediction Explanation) ---
# Explains WHY the model made a specific prediction for one company

# Pick a bankrupt company from test set to explain
bankrupt_idx = np.where(y_test.values == 1)[0]
if len(bankrupt_idx) > 0:
    sample_idx = bankrupt_idx[0]
    print(f'Explaining prediction for test sample index: {sample_idx}')
    print(f'True label: {"Bankrupt" if y_test.values[sample_idx] == 1 else "Healthy"}')
    print(f'Predicted:  {"Bankrupt" if results[best_model_name]["y_pred"][sample_idx] == 1 else "Healthy"}')
    
    # Waterfall plot
    shap_exp = shap.Explanation(
        values=sv[sample_idx],
        base_values=explainer.expected_value if not isinstance(explainer.expected_value, list)
                    else explainer.expected_value[1],
        data=X_test.iloc[sample_idx].values,
        feature_names=X_test.columns.tolist()
    )
    shap.waterfall_plot(shap_exp, show=True)

In [ ]:
# --- SHAP Force Plot (Interactive HTML — works in Jupyter) ---
shap.initjs()
force_plot = shap.force_plot(
    explainer.expected_value if not isinstance(explainer.expected_value, list)
    else explainer.expected_value[1],
    sv[sample_idx],
    X_test.iloc[sample_idx],
    feature_names=X_test.columns.tolist()
)
force_plot

In [ ]:
print('=' * 55)
print('         BANKRUPTCY PREDICTION — FINAL SUMMARY')
print('=' * 55)
print(f'Dataset       : Polish Companies 5year.arff')
print(f'Samples       : {df.shape[0]} companies')
print(f'Features used : {X_selected.shape[1]} (selected from {X_raw.shape[1]})')
print(f'Class balance : {dict(y.value_counts())}')
print(f'Imbalance fix : SMOTE oversampling')
print('-' * 55)
print('Model Performance on Test Set:')
for name, res in results.items():
    marker = ' 🏆' if name == best_model_name else ''
    print(f'  {name:<25} F1={res["f1"]:.4f}  AUC={res["roc_auc"]:.4f}{marker}')
print('-' * 55)
print(f'Best Model    : {best_model_name}')
print(f'XAI Method    : SHAP (TreeExplainer)')
print('=' * 55)